In [1]:
import pandas as pd

df=pd.read_json('liquipedia_data/clean_data/fortnite/players.json')

In [2]:
df[df["tier"]=="medium"]

,pageid,pagename,id,alternateid_list,name,type,status,nationalities,region,birthdate,...,earnings_2019,earnings_2020,earnings_2021,earnings_2022,earnings_2023,earnings_2024,earnings_2025,earnings_2026,fncs_wins,tier
6,10541,1lusha,1lusha,[],NaN,player,Active,[Russia],Europe,2004-07-22,...,2520,4089,48533,26438,13598,12592,16823,18650,1,medium
12,11918,2SNgNl,2SNgNl,[],NaN,player,Retired,[Russia],Europe,2003-06-10,...,1925,8455,17120,10125,0,0,0,0,0,medium
25,14223,3vil,3vil,[],Oskar Żeljazkow,player,Retired,[Poland],Europe,1988-08-24,...,23460,0,0,800,0,0,0,0,0,medium
29,8530,4DRStorm,4DRStorm,[],NaN,player,Retired,[United States],North America,2003-05-23,...,128358,22850,14020,4825,900,0,0,0,0,medium
30,17502,4Turtle,4Turtle,[],NaN,player,Retired,[Australia],Oceania,NaN,...,63750,5354,0,0,500,2500,0,0,0,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5719,19652,Zynox,Zynox,[],NaN,player,Active,[United Kingdom],Europe,NaN,...,0,0,0,1400,5950,25225,15925,25088,0,medium
5720,14924,Zyppan,Zyppan,[Zyppaan],Pontus Eek,player,Retired,[Sweden],Europe,2002-07-17,...,19838,0,0,0,0,0,0,0,0,medium
5721,14280,Zyrofnw,Zyrofnw,[],Hector Gael Garcia Vizcarra,player,Active,[Mexico],North America,2007-07-02,...,0,0,0,5050,6850,4200,47450,1800,0,medium
5723,14548,せこさま,Phazma,[SEKO],NaN,player,Retired,[Japan],Asia,NaN,...,11575,4505,3070,13000,10593,0,0,0,0,medium


In [3]:
import json, re, unicodedata
from collections import Counter, defaultdict

PLAYERS = 'liquipedia_data/clean_data/fortnite/players.json'
LEGACY  = 'src/data/fortnite'          # the Wikipedia import

rows = json.load(open(PLAYERS, encoding='utf-8'))
print(len(rows), 'players')

5726 players


In [2]:
def norm(s):
    """Match key for a handle: fold accents, keep a-z0-9."""
    s = unicodedata.normalize('NFKD', s or '')
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]', '', s.lower())

def base(s):
    """'Speedy (BH)' -> 'speedy'"""
    return norm(re.sub(r'[_ ]*\(.*', '', (s or '').replace('_', ' ')))

def lines(name):
    return open(f'{LEGACY}/{name}', encoding='utf-8').read().splitlines()

tiers = dict(m.groups() for m in
             (re.search(r"id: '([^']+)'.*?tier: '([^']+)'", l) for l in lines('events.ts')) if m)

legacy = {}
for l in lines('players.ts'):
    m = re.search(r"id: '([^']+)', name: '([^']*)'.*?country: '([^']+)'", l)
    if m:
        legacy[m.group(1)] = (m.group(2), m.group(3))

wiki_wins = Counter()
for l in lines('entries.ts'):
    m = re.search(r"eventId: '([^']+)', placement: (\d+), playerIds: \[([^\]]*)\]", l)
    if not m or int(m.group(2)) != 1 or tiers.get(m.group(1)) != 'fncs':
        continue
    for pid in re.findall(r"'([^']+)'", m.group(3)):
        wiki_wins[legacy.get(pid, (pid, None))] += 1

index = defaultdict(list)
for r in rows:
    for k in {norm(r['id']), norm(r['pagename']),
              *(norm(a) for a in (r.get('alternateid_list') or []))}:
        if k:
            index[k].append(r)

CODE_TO_NAT = {
    'US': 'United States', 'CA': 'Canada', 'GB': 'United Kingdom', 'AU': 'Australia',
    'JP': 'Japan', 'BR': 'Brazil', 'FR': 'France', 'DE': 'Germany', 'SA': 'Saudi Arabia',
    'MX': 'Mexico', 'PL': 'Poland', 'RU': 'Russia', 'AT': 'Austria', 'SE': 'Sweden',
    'DK': 'Denmark', 'NL': 'Netherlands', 'NO': 'Norway', 'IE': 'Ireland', 'IT': 'Italy',
    'ES': 'Spain', 'PT': 'Portugal', 'LT': 'Lithuania', 'LV': 'Latvia', 'SI': 'Slovenia',
    'RS': 'Serbia', 'HR': 'Croatia', 'BA': 'Bosnia and Herzegovina', 'UA': 'Ukraine',
    'KR': 'South Korea', 'SG': 'Singapore', 'MY': 'Malaysia', 'IN': 'India',
    'ID': 'Indonesia', 'PK': 'Pakistan', 'AE': 'United Arab Emirates', 'BH': 'Bahrain',
    'KW': 'Kuwait', 'JO': 'Jordan', 'OM': 'Oman', 'SY': 'Syria', 'CL': 'Chile',
    'AR': 'Argentina', 'CU': 'Cuba', 'NZ': 'New Zealand',
}

ALIASES = {
    'Kalgamer': 'Kalgamer710',
    'Kiryache': 'Kiryache32',
    'Speedy (BH)': 'Speedy',
    'Bobi': 'Bobik1ng',        # id is 'BOBY', page is 'Bobik1ng'
    'Buyuriro': 'Buyuriru',
    'Drobban': 'Drobbаn',      # NB: Cyrillic 'а' — see below
    'Kucha': 'Kocha',
    'Mansoor': 'Mansour',
    'Murlox': 'Murloc',
    'Takamura': 'Ruri',
}

def resolve(handle, country=None):
    hits = index.get(norm(handle)) or index.get(base(handle))
    if not hits:
        return None
    if len(hits) > 1 and country:                       # 142 handles are ambiguous
        same = [r for r in hits if CODE_TO_NAT.get(country) in (r.get('nationalities') or [])]
        if same:
            hits = same
    return max(hits, key=lambda r: r.get('earnings') or 0)

fncs, unmatched = {}, []
for (handle, country), n in wiki_wins.items():
    r = resolve(ALIASES.get(handle, handle), country)
    if r is None:
        unmatched.append(handle)
    else:
        fncs[r['pagename']] = max(fncs.get(r['pagename'], 0), n)

for r in rows:
    r['fncs_wins'] = fncs.get(r['pagename'], 0)

print(len(fncs), 'players given a title count')
print('no Liquipedia page:', ', '.join(sorted(unmatched)))

284 players given a title count
no Liquipedia page: 


In [4]:
EASY_PCT, MEDIUM_PCT = 0.02, 0.20     # top 2% easy, next 18% medium, rest hard

def unusable(r):
    """Rows the games must never touch. This is the bit you'll rewrite."""
    return (r.get('status') or '').lower() == 'passed away' or (r.get('earnings') or 0) <= 0

usable = [r for r in rows if not unusable(r)]
usable.sort(key=lambda r: (-(r.get('earnings') or 0), r['id'].lower()))

easy_cut, med_cut = len(usable) * EASY_PCT, len(usable) * MEDIUM_PCT
for r in rows:
    r['tier'] = 'unused'
for i, r in enumerate(usable, start=1):
    r['tier'] = 'easy' if i <= easy_cut else 'medium' if i <= med_cut else 'hard'

print(Counter(r['tier'] for r in rows))

Counter({'hard': 4543, 'medium': 1022, 'easy': 113, 'unused': 48})


In [4]:
# separators=(',', ':') with indent=2 matches the file's existing style
with open(PLAYERS, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2, separators=(',', ':'))

check = json.load(open(PLAYERS, encoding='utf-8'))
assert all(r['tier'] in {'easy', 'medium', 'hard', 'unused'} for r in check)
assert all(isinstance(r['fncs_wins'], int) and r['fncs_wins'] >= 0 for r in check)
print('saved', len(check), 'rows')

saved 5726 rows


In [5]:
# region_tier — the same cut as `tier`, ranked within each region.
#
# Fortnitedle lets you pick a region first. Global `tier` cannot serve that:
# it ranks all 5,678 rows on earnings, so Asia, Oceania and the Middle East
# have zero Easy players between them and "Asia + Easy" would quietly hand
# back Medium instead. Ranked inside the region, every region has all three.
#
# `unused` carries over unchanged — one rule for who is playable, not two.
from collections import Counter, defaultdict

by_region = defaultdict(list)
for r in rows:
    if r['tier'] != 'unused':
        by_region[r.get('region') or 'Unknown'].append(r)

for r in rows:
    r['region_tier'] = 'unused'

for region, members in by_region.items():
    members.sort(key=lambda r: (-(r.get('earnings') or 0), r['id'].lower()))
    easy_cut, med_cut = len(members) * EASY_PCT, len(members) * MEDIUM_PCT
    for i, r in enumerate(members, start=1):
        r['region_tier'] = 'easy' if i <= easy_cut else 'medium' if i <= med_cut else 'hard'

for region, members in sorted(by_region.items(), key=lambda kv: -len(kv[1])):
    c = Counter(r['region_tier'] for r in members)
    print(f"{region:<16} easy {c['easy']:>4}  medium {c['medium']:>4}  hard {c['hard']:>5}")

North America    easy   36  medium  332  hard  1473
Europe           easy   32  medium  295  hard  1311
Asia             easy   13  medium  118  hard   526
Oceania          easy   12  medium  112  hard   500
South America    easy    9  medium   83  hard   370
Middle East      easy    8  medium   78  hard   348
Africa           easy    0  medium    4  hard    18


In [6]:
# separators=(',', ':') with indent=2 matches the file's existing style
with open(PLAYERS, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2, separators=(',', ':'))

check = json.load(open(PLAYERS, encoding='utf-8'))
assert all(r['region_tier'] in {'easy', 'medium', 'hard', 'unused'} for r in check)
print('saved', len(check), 'rows')

saved 5726 rows


In [5]:
import pandas as pd

df=pd.read_json(PLAYERS)

In [6]:
df.head()

,pageid,pagename,id,alternateid_list,name,type,status,nationalities,region,birthdate,...,earnings_2019,earnings_2020,earnings_2021,earnings_2022,earnings_2023,earnings_2024,earnings_2025,earnings_2026,fncs_wins,tier
0,4814,00flour,00flour,[],Ryan Borst,player,Retired,[United States],North America,1996-09-29,...,125,0,0,0,0,0,0,0,0,hard
1,29814,10bit,10bit,[10bittt],NaN,player,Retired,[Canada],North America,NaN,...,6350,0,0,0,0,0,0,0,0,hard
2,32523,10m,10m,[],NaN,player,Active,[Bahrain],Middle East,NaN,...,0,0,0,200,1350,1000,700,100,0,hard
3,19346,13,13,[],NaN,player,Retired,[China],Asia,NaN,...,8125,0,0,0,0,0,0,0,0,hard
4,28922,1Saud,1Saud,[],NaN,player,Active,[Saudi Arabia],Middle East,NaN,...,0,0,0,0,0,700,2530,0,0,hard


In [7]:
df[df["tier"]=="easy"]

,pageid,pagename,id,alternateid_list,name,type,status,nationalities,region,birthdate,...,earnings_2019,earnings_2020,earnings_2021,earnings_2022,earnings_2023,earnings_2024,earnings_2025,earnings_2026,fncs_wins,tier
41,703,72hrs,72hrs,[],Thomas Mulligan,player,Retired,[United States],North America,1994-05-26,...,40625,1133,0,0,0,0,0,0,0,easy
91,8526,Acorn,Acorn,[],Abdullah Akhras,player,Active,[Canada],North America,2004-06-14,...,15000,72326,165488,138043,224790,187150,187725,129600,5,easy
149,8855,Ajerss,Ajerss,[],Aidan Joseph Bernero,player,Active,[United States],North America,2005-11-19,...,10700,28830,113935,75188,168625,62000,158650,69500,2,easy
213,8539,Anas,Anas,[],Anas El-Abd,player,Active,[Denmark],Europe,2002-12-11,...,26900,75066,255379,1242319,29940,27763,1450,850,0,easy
218,8174,Andilex,Andilex,[],Alexandre Christophe,player,Active,[France],Europe,2003-01-31,...,81550,186308,152144,78258,25947,56219,2425,0,1,easy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5397,8163,Wolfiez,Wolfiez,[],Jaden Ashman,player,Retired,[United Kingdom],Europe,2003-12-11,...,1194017,169239,5015,950,0,0,0,0,0,easy
5559,11026,ZAndy,zAndy,[],Andrei Ursu,player,Active,[Romania],Europe,2006-01-17,...,4310,7600,8655,205600,90128,3450,3150,450,0,easy
5585,15053,Zand,Zand,[],Martin Fløjgaard,player,Retired,[Denmark],Europe,NaN,...,341250,0,0,0,0,0,0,0,0,easy
5600,2693,Zayt,Zayt,[],Williams Aubin,player,Retired,[Canada],North America,2000-03-02,...,959525,81003,12377,0,0,0,0,0,1,easy


In [7]:
import json, re
from datetime import date
import pandas as pd

OUT = 'liquipedia_data/clean_data/fortnite/career_path.json'
MIN_APPEARANCES = 5          # a path is only worth guessing from this many majors

df_tournaments = pd.read_json('liquipedia_data/clean_data/fortnite/tournaments.json')

def has_epic_games(val):
    if isinstance(val, list):
        return any(
            "epic games" in str(item).lower()
            or (isinstance(item, dict) and "epic games" in str(item.values()).lower())
            for item in val
        )
    elif isinstance(val, str):
        return "epic games" in val.lower()
    return False

exclude_pattern = r"console|mobile|twitch|challenge"

filtered_df = df_tournaments[
    (df_tournaments["liquipediatier"] == 1)
    & (df_tournaments["liquipediatiertype"].isna())
    & (df_tournaments["organizers"].apply(has_epic_games))
    & (pd.to_datetime(df_tournaments["startdate"], errors="coerce") >= "2019-07-26")
    & (~df_tournaments["name"].str.contains(exclude_pattern, case=False, na=False))
].sort_values("startdate")

print(len(filtered_df), 'tournaments')

# ---------------------------------------------------------------- tournaments
tournaments, index_of = [], {}
for _, t in filtered_df.iterrows():
    index_of[t['name']] = len(tournaments)
    tournaments.append({
        'name': t['name'],
        'date': str(t['startdate'])[:10],
        'mode': t['mode'],
        'region': t['region'],
        'prizePool': None if pd.isna(t['prizepool']) else round(float(t['prizepool'])),
    })

# ------------------------------------------------------------------ placements
players_rows = json.load(open('liquipedia_data/clean_data/fortnite/players.json', encoding='utf-8'))
by_page = {p['pagename'].replace('_', ' '): p['pagename'] for p in players_rows}
by_id = {}
for p in players_rows:
    by_id.setdefault(p['id'], p['pagename'])

def resolve(name):
    """A placement's participant name -> a players.json pagename, or None."""
    return by_page.get(name) or by_id.get(name)

placements = json.load(open('liquipedia_data/clean_data/fortnite/placements.json', encoding='utf-8'))

results = {}
for row in placements:
    idx = index_of.get(row.get('tournament'))
    if idx is None:
        continue
    # '', 'DNP' and 'DQ' are not a finish anyone can be identified by; a range
    # like '35-36' is, and reads as its best end.
    m = re.match(r'^(\d+)', str(row.get('placement') or ''))
    if not m:
        continue
    placement = int(m.group(1))
    for part in (row.get('participants') or []):
        page = resolve(part.get('player') or '')
        if page:
            results.setdefault(page, {})[idx] = placement

players = [
    {'id': page, 'results': sorted([i, p] for i, p in hits.items())}
    for page, hits in results.items()
    if len(hits) >= MIN_APPEARANCES
]
players.sort(key=lambda p: p['id'])

payload = {
    'generated': date.today().isoformat(),
    'minAppearances': MIN_APPEARANCES,
    'tournaments': tournaments,
    'players': players,
}
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, separators=(',', ':'))

tier = {p['pagename']: p['tier'] for p in players_rows}
from collections import Counter
print(len(players), 'players with', MIN_APPEARANCES, '+ appearances')
print(Counter(tier.get(p['id']) for p in players))
print('longest path:', max(len(p['results']) for p in players))

187 tournaments
1175 players with 5 + appearances
Counter({'medium': 567, 'hard': 516, 'easy': 92})
longest path: 34


In [10]:
import json, itertools
from collections import Counter, defaultdict
from datetime import date

OUT = 'liquipedia_data/clean_data/fortnite/teammates.json'
TOP = 10          # Who Are Ya reveals at most this many clues

players_rows = json.load(open('liquipedia_data/clean_data/fortnite/players.json', encoding='utf-8'))
by_page = {p['pagename'].replace('_', ' '): p['pagename'] for p in players_rows}
by_id = {}
for p in players_rows:
    by_id.setdefault(p['id'], p['pagename'])

def resolve(name):
    return by_page.get(name) or by_id.get(name)

placements = json.load(open('liquipedia_data/clean_data/fortnite/placements.json', encoding='utf-8'))

pair = Counter()
for row in placements:
    parts = row.get('participants') or []
    if len(parts) < 2:
        continue
    pages = sorted({resolve(p.get('player') or '') for p in parts} - {None})
    for a, b in itertools.combinations(pages, 2):
        pair[(a, b)] += 1

mates = defaultdict(list)
for (a, b), n in pair.items():
    mates[a].append([b, n])
    mates[b].append([a, n])

players = []
for page, entries in mates.items():
    entries.sort(key=lambda e: (-e[1], e[0]))
    players.append({'id': page, 'mates': entries[:TOP]})
players.sort(key=lambda p: p['id'])

payload = {'generated': date.today().isoformat(), 'top': TOP, 'players': players}
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, separators=(',', ':'))

tier = {p['pagename']: p['tier'] for p in players_rows}
print(len(pair), 'distinct pairs;', len(players), 'players with a teammate')
print('>=3 teammates:', Counter(tier.get(p['id']) for p in players if len(p['mates']) >= 3))
print('Peterbot:', next(p for p in players if p['id'] == 'Peterbot')['mates'])

39038 distinct pairs; 5496 players with a teammate
>=3 teammates: Counter({'hard': 3692, 'medium': 1000, 'easy': 113, 'unused': 3})
Peterbot: [['Pollo', 126], ['Cold', 54], ['Ritual', 49], ['Bugha', 27], ['Kwanti', 27], ['Bylah', 23], ['Larson', 18], ['MackWood', 18], ['Bucke', 17], ['Faxuty', 15]]


In [11]:
import json
from collections import defaultdict
from datetime import date

OUT = 'liquipedia_data/clean_data/fortnite/orgs.json'
MIN_PLAYERS = 4          # a criterion nobody can fill is not a criterion

BASE = 'liquipedia_data/clean_data/fortnite'
players_rows = json.load(open(f'{BASE}/players.json', encoding='utf-8'))
teams_rows = json.load(open(f'{BASE}/teams.json', encoding='utf-8'))
transfers = json.load(open(f'{BASE}/transfers.json', encoding='utf-8'))

by_page = {p['pagename'].replace('_', ' '): p['pagename'] for p in players_rows}
by_id = {}
for p in players_rows:
    by_id.setdefault(p['id'], p['pagename'])
tier = {p['pagename']: p['tier'] for p in players_rows}

def resolve(name):
    """A transfer's player name -> a playable players.json pagename, or None."""
    page = by_page.get(name) or by_id.get(name)
    return page if page and tier.get(page) != 'unused' else None

page_of = {t['name']: t['pagename'] for t in teams_rows}
team_meta = {t['pagename']: t for t in teams_rows}

NOT_A_TEAM = {'free agent', 'retired', 'retirement', 'inactive', 'none', 'unknown', ''}

def org_key(name):
    """Transfers spell orgs by display name; teams.json and players.json use the
    page name. Falls back to the raw string for the ~2,100 grassroots orgs with
    no Liquipedia team page — `hasPage` below says which is which."""
    if not name or name.strip().lower() in NOT_A_TEAM:
        return None
    return page_of.get(name, name)

ever = defaultdict(set)
for row in transfers:
    page = resolve(row.get('player') or '')
    if not page:
        continue
    # Per side, not per row: `role_from` is what they were at `fromteam` and
    # `role_to` what they became at `toteam`, so someone who left as a player
    # and joined as a streamer counts for the first org and not the second.
    for side, role in (('fromteam', 'role_from'), ('toteam', 'role_to')):
        if row.get(role) != 'Player':
            continue
        key = org_key(row.get(side))
        if key:
            ever[key].add(page)

# players.json is the authority on who is there now, and catches recent
# signings that have no transfer row yet.
current = defaultdict(set)
for p in players_rows:
    if p['tier'] != 'unused' and p.get('teampagename'):
        current[p['teampagename']].add(p['pagename'])
for key, members in current.items():
    ever[key] |= members

orgs = []
for key, members in ever.items():
    if len(members) < MIN_PLAYERS:
        continue
    meta = team_meta.get(key)
    orgs.append({
        'id': key,
        'name': meta['name'] if meta else key,
        'hasPage': meta is not None,
        'region': (meta or {}).get('region'),
        'status': (meta or {}).get('status'),
        'earnings': round(float((meta or {}).get('earnings') or 0)),
        'current': sorted(current.get(key, ())),
        'ever': sorted(members),
    })
# Richest first, so "take the top N" is a one-liner later.
orgs.sort(key=lambda o: (-o['earnings'], o['name'].lower()))

payload = {'generated': date.today().isoformat(), 'minPlayers': MIN_PLAYERS, 'orgs': orgs}
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, separators=(',', ':'))

paged = [o for o in orgs if o['hasPage']]
print(f'{len(orgs)} orgs with {MIN_PLAYERS}+ players, {len(paged)} with a Liquipedia team page')
print('  with page and $100k+ org earnings:', sum(1 for o in paged if o['earnings'] >= 100_000))
print('  with page and $1M+ org earnings:  ', sum(1 for o in paged if o['earnings'] >= 1_000_000))
print('  with a current roster of 4+:      ', sum(1 for o in orgs if len(o['current']) >= MIN_PLAYERS))
for o in orgs[:10]:
    print(f"  {o['name']:<24} ever {len(o['ever']):>3}  now {len(o['current']):>2}  ${o['earnings']:,}")

979 orgs with 4+ players, 414 with a Liquipedia team page
  with page and $100k+ org earnings: 174
  with page and $1M+ org earnings:   30
  with a current roster of 4+:       69
  FaZe Clan                ever  21  now  0  $4,635,700
  NRG                      ever  12  now  1  $4,592,147
  Sentinels                ever   8  now  0  $4,114,758
  Team Falcons             ever  23  now  7  $3,764,580
  Lazarus                  ever   9  now  0  $3,714,968
  100 Thieves              ever  21  now  7  $3,713,116
  Guild Esports            ever  14  now  0  $3,206,091
  Ghost Gaming             ever  20  now  1  $3,068,598
  Team Liquid              ever  20  now  4  $2,644,750
  XSET                     ever  22  now 13  $2,393,596
